# E03: Métodos Dunder y Protocolos de Python

**Nivel:** Intermedio

Los **métodos dunder** (del inglés *double underscore*, `__método__`) son el conjunto de métodos especiales que Python reconoce por convención para integrar tus clases con el propio lenguaje. Son la base de los **protocolos** de Python: conjuntos informales de métodos que, si implementas, tu objeto se comporta como un tipo nativo.

En este notebook aprenderás a:
- Representar tus objetos (`__repr__`, `__str__`, `__format__`)
- Hacer que tus objetos se comporten como secuencias y contenedores
- Crear iterables e iteradores
- Controlar igualdad, hash y *truthiness*
- Implementar el protocolo *context manager* (`with`)
- Hacer tus objetos llamables (`()`), sumables y operables con operadores

## Objetivos

Al finalizar este notebook serás capaz de:

1. **Explicar** qué son los métodos dunder y por qué Python los invoca automáticamente.
2. **Distinguir** entre `__repr__` y `__str__`, y cuándo usar cada uno.
3. **Implementar** el protocolo de secuencia (`__len__`, `__getitem__`, etc.) para que una clase admita `len()`, indexado, `in` e iteración.
4. **Construir** iterables e iteradores propios con `__iter__` y `__next__`, entendiendo la diferencia entre ambos.
5. **Aplicar** protocolos avanzados: igualdad/hash, *context managers*, objetos llamables y operadores aritméticos.

## Analogía: Señales de tráfico y contratos informales

Imagina una ciudad donde las calles están llenas de **señales de tráfico**. Los conductores (el intérprete de Python) no necesitan que cada cruce esté vigilado: solo miran las señales y actúan automáticamente según lo que ven.

- Un **semáforo** (`__len__`) indica a Python "sé cuántos elementos tengo", así que `len(obj)` funciona.
- Una **señal de ceda el paso** (`__contains__`) le dice "revisa si un elemento está en mí", así que `x in obj` funciona.
- Una **flecha de ruta** (`__iter__`) le indica "puedes recorrerme", así que `for elem in obj` funciona.

Los **protocolos** son **contratos informales**: no hay una clase base obligatoria que heredar, solo la promesa de implementar ciertos métodos. Si un objeto "camina como un pato y suena como un pato" (tiene los métodos esperados), Python lo trata como un pato. Esto se llama **duck typing**.

Cuando definas `__len__`, Python automáticamente:
```python
len(obj)   # llama a obj.__len__()
bool(obj)  # si no hay __bool__, usa len(obj) != 0
```

No necesitas registrar nada: **el método dunder correcto crea el contrato**. Eso es el poder de los protocolos.

## 1. Representación: `__repr__` vs `__str__`

Python tiene **dos** métodos para convertir un objeto en texto. Ambos se llaman automáticamente, pero en contextos distintos:

| Método | Propósito | Cuándo lo llama Python | Audiencia |
|--------|-----------|------------------------|-----------|
| `__str__` | Representación legible | `print(obj)`, `f"{obj}"`, `str(obj)` | Usuarios finales |
| `__repr__` | Representación sin ambigüedad / para debug | `repr(obj)`, REPL, f"{obj!r}" | Programadores |

Regla práctica: `__repr__` debería ser **sin ambigüedad** (idealmente reconstruir el objeto), y `__str__` debería ser **legible** para humanos. Si solo defines uno, define `__repr__`.

In [ ]:
class Punto:
    def __init__(self, x: int, y: int) -> None:
        self.x = x
        self.y = y

    def __repr__(self) -> str:
        # Sin ambigüedad: se puede reconstruir con eval
        return f"Punto({self.x!r}, {self.y!r})"

    def __str__(self) -> str:
        # Legible para humanos
        return f"({self.x}, {self.y})"

p = Punto(3, 5)

print("str  ->", str(p))
print("repr ->", repr(p))
print(f"f-string -> {p}   |   con !r -> {p!r}")

`__format__` permite personalizar cómo se formatea el objeto dentro de un f-string o `format()`:

```python
def __format__(self, spec: str) -> str:
    # spec es lo que va después de los dos puntos en {obj:spec}
    ...
```

In [ ]:
class Temperatura:
    def __init__(self, celsius: float) -> None:
        self.celsius = celsius

    def __repr__(self) -> str:
        return f"Temperatura({self.celsius!r})"

    def __format__(self, spec: str) -> str:
        if spec == "far":
            far = self.celsius * 9 / 5 + 32
            return f"{far:.1f} °F"
        if spec == "kel":
            return f"{self.celsius + 273.15:.1f} K"
        return f"{self.celsius:.1f} °C"

t = Temperatura(25)
print(f"Celsius: {t}")
print(f"Fahrenheit: {t:far}")
print(f"Kelvin: {t:kel}")

## 2. Protocolo de secuencia

Las secuencias y contenedores nativos (`list`, `tuple`, `dict`, `str`) comparten un **protocolo de contenedor**. Al implementarlo, tu clase se comporta como un contenedor real:

| Método | Función que activa |
|--------|--------------------|
| `__len__` | `len(obj)` |
| `__getitem__` | `obj[i]`, slicing, iteración |
| `__setitem__` | `obj[i] = valor` |
| `__delitem__` | `del obj[i]` |
| `__contains__` | `x in obj` |

### Iteración automática (secuencia legada)

Un detalle clave: **si una clase define `__len__` y `__getitem__` pero no `__iter__`**, Python la itera como una *secuencia legada*: internamente llama a `__getitem__(0)`, `__getitem__(1)`, ... hasta que se lanza `IndexError`. Recorre así sin necesidad de un iterador explícito.

In [ ]:
class Libro:
    def __init__(self, titulo: str, capitulos: list[str]) -> None:
        self.titulo = titulo
        self._capitulos = capitulos

    def __len__(self) -> int:
        return len(self._capitulos)

    def __getitem__(self, indice: int | slice):
        return self._capitulos[indice]

    def __setitem__(self, indice: int, valor: str) -> None:
        self._capitulos[indice] = valor

    def __delitem__(self, indice: int) -> None:
        del self._capitulos[indice]

    def __contains__(self, capitulo: str) -> bool:
        return capitulo.lower() in map(str.lower, self._capitulos)

    def __repr__(self) -> str:
        return f"Libro({self.titulo!r}, {self._capitulos!r})"

libro = Libro("Python Avanzado", ["Intro", "Dunder", "Protocolos", "ML"])

print("len:", len(libro))
print("index 0:", libro[0])
print("slice:", libro[1:3])
print("in (case-insensitive):", "dunder" in libro)
print("in (falso):", "web" in libro)

libro[3] = "Machine Learning"      # __setitem__
del libro[0]                        # __delitem__
print("tras modificar:", libro)

In [ ]:
# Iteración automática mediante __len__ + __getitem__ (secuencia legada)
for cap in libro:
    print("- ", cap)

print("---")
for i, cap in enumerate(libro):
    print(i, cap)

## 3. Iterator / Protocolo de iteración

Cuando una secuencia legada no basta (porque quieres secuencias *infinitas*, perezosas o con estado), necesitas el protocolo de iteración moderno. Aquí aparece una distinción fundamental:

- **Iterable**: un objeto que implementa `__iter__` (devuelve un iterador). Puede ser recorrido con `for` varias veces.
- **Iterador**: un objeto que implementa `__iter__` (devuelve a sí mismo) y `__next__` (devuelve el siguiente elemento o lanza `StopIteration`). Es de **un solo uso**.

### Diagrama ASCII del protocolo de iteración

```
            for x in iterable:
                  |
                  v
         +--------------------+
         |      ITERABLE      |   implementa __iter__
         |  __iter__() -> it  |   devuelve un iterador
         +--------------------+
                  |
         iter(obj)|
                  v
         +--------------------+
         |      ITERADOR      |   implementa __iter__ y __next__
         |  __iter__() -> self|
         |  __next__() -> x   |   siguiente elemento
         |  __next__() -> Stop|
         +--------------------+
                  |
     excepcion StopIteration
                  v
        el bucle for termina

Resumen:
    iterable.__iter__()   -> iterador
    iterador.__next__()   -> elemento  (o StopIteration)
    todo iterador es iterable; no todo iterable es iterador.
```

La forma más limpia es que una clase sea **simultáneamente iterable e iterador** (devuelve `self` en `__iter__`), útil para secuencias con estado como generadores manuales.

In [ ]:
class CuentaRegresiva:
    """Iterable e iterador a la vez (caso con estado)."""
    def __init__(self, inicio: int) -> None:
        self.actual = inicio

    def __iter__(self):
        return self          # soy mi propio iterador

    def __next__(self) -> int:
        if self.actual < 0:
            raise StopIteration
        valor = self.actual
        self.actual -= 1
        return valor

    def __repr__(self) -> str:
        return f"CuentaRegresiva({self.actual})"

# __next__ manual
cr = CuentaRegresiva(3)
print(next(cr))   # 3
print(next(cr))   # 2
print(next(cr))   # 1
print(next(cr))   # 0
# print(next(cr))  # -> StopIteration

In [ ]:
# El bucle for gestiona la excepción StopIteration automáticamente
for numero in CuentaRegresiva(3):
    print(numero)

print("--- list(iterable) ---")
print(list(CuentaRegresiva(2)))

En muchos casos no necesitas escribir un iterador a mano: el **generador** (`yield`) construye el iterador automáticamente.

```python
def pares(maximo):
    for n in range(0, maximo + 1, 2):
        yield n
```

Diferencia clave entre un iterable y un iterador:

| Característica | Iterable | Iterador |
|----------------|----------|----------|
| Método requerido | `__iter__` | `__iter__` + `__next__` |
| Puede recorrerlo varias veces | Sí* | No (un solo uso) |
| Ejemplos | `list`, `tuple`, `str`, `range` | `iterator` de `iter()`, generadores |

\* Salvo que sea su propio iterador con estado.

In [ ]:
# Separación iterable/iterador: list es iterable, no iterador
datos = [1, 2, 3]
print("iter(datos):", iter(datos))
print("tiene __next__? list:", hasattr(datos, "__next__"))
it = iter(datos)
print("tiene __next__? iterador:", hasattr(it, "__next__"))
print("next(it):", next(it))
print("next(it):", next(it))
print("next(it):", next(it))
# print(next(it))  # -> StopIteration

## 4. Igualdad, hash y truthiness

Para comparar objetos con `==`, `<`, etc., y usarlos como claves de diccionario o en conjuntos, debes implementar el protocolo de igualdad y hash.

| Método | Función que activa |
|--------|--------------------|
| `__eq__` | `a == b` (y con `@total_ordering`, los demás) |
| `__hash__` | `hash(a)`, uso en `set`/`dict` |
| `__lt__` | `a < b` (con `functools.total_ordering` genera el resto) |
| `__bool__` | `bool(a)` y `if a:` |

### Reglas importantes

1. **`__eq__` y `__hash__` van de la mano**: si defines `__eq__` sin `__hash__`, Python pone `__hash__ = None` (el objeto se vuelve *unhashable*), porque objetos iguales deben tener el mismo hash, y mutables no deberían ser hashables.
2. **Hashable** = se puede usar como clave de dict / elemento de set. Las tuplas y cadenas lo son; las listas y dicts no.
3. **Truthiness**: si no hay `__bool__` ni `__len__`, un objeto es siempre *truthy*. Con `__len__`, `bool(obj)` es `len(obj) != 0`.

In [ ]:
from functools import total_ordering

@total_ordering
class Moneda:
    def __init__(self, valor: int) -> None:
        self.valor = valor

    def __repr__(self) -> str:
        return f"Moneda({self.valor})"

    def __eq__(self, otro) -> bool:
        if not isinstance(otro, Moneda):
            return NotImplemented
        return self.valor == otro.valor

    def __lt__(self, otro) -> bool:
        if not isinstance(otro, Moneda):
            return NotImplemented
        return self.valor < otro.valor

    def __hash__(self) -> int:
        return hash(self.valor)

m1 = Moneda(5)
m2 = Moneda(5)
m3 = Moneda(10)

print("m1 == m2:", m1 == m2)
print("m1 < m3:", m1 < m3)
print("m3 > m1 (desde total_ordering):", m3 > m1)
print("m1 <= m2 (desde total_ordering):", m1 <= m2)

# Hashable -> usable en conjunto y dict
monedas = {m1, m2, m3}
print("set único (dedupe por igualdad):", monedas)

In [ ]:
# Un objeto mutable con __eq__ pero sin __hash__ se vuelve unhashable
class CajaMutable:
    def __init__(self, items: list[str]) -> None:
        self.items = items

    def __eq__(self, otro) -> bool:
        return isinstance(otro, CajaMutable) and self.items == otro.items

    # NO se define __hash__ -> Python lo pone a None

c = CajaMutable(["a"])
print("__hash__ es:", CajaMutable.__hash__)  # None
print("es hashable?", hash(c) if CajaMutable.__hash__ else "NO (unhashable)")

In [ ]:
# Truthiness: __bool__ + __len__, y su prioridad
class Fila:
    def __init__(self, elementos: list[int]) -> None:
        self.elementos = elementos

    def __len__(self) -> int:
        return len(self.elementos)

    def __bool__(self) -> bool:
        # __bool__ tiene prioridad sobre __len__
        return sum(self.elementos) > 0

print("bool(Fila([1,-2])):", bool(Fila([1, -2])))   # suma -1 -> False
print("bool(Fila([5])):", bool(Fila([5])))         # suma 5  -> True

# Si no existiera __bool__, se usaría len
class SoloLen:
    def __init__(self, n):
        self.n = n
    def __len__(self):
        return self.n

print("bool(SoloLen(0)):", bool(SoloLen(0)))
print("bool(SoloLen(3)):", bool(SoloLen(3)))

## 5. Context manager protocol

El protocolo *context manager* hace que tu objeto funcione con la instrucción `with`, gestionando automáticamente la **adquisición y liberación de recursos** (archivos, conexiones, locks).

| Método | Cuándo se llama |
|--------|-----------------|
| `__enter__` | Al entrar en el bloque `with` (devuelve lo que se asigna a `as`) |
| `__exit__` | Al salir del bloque, incluso si hay excepción (recibe el error o `None` si todo fue bien) |

La firma de `__exit__`:
```python
def __exit__(self, exc_type, exc_val, exc_tb) -> bool | None:
    ...
```

- Si devuelve `True`, **suprime** la excepción que se hubiera lanzado.
- Si devuelve falsy (o `None`), la excepción **se propaga**.
    - `exc_type`: tipo de la excepción (`ZeroDivisionError`, ...)
    - `exc_val`: la instancia de la excepción
    - `exc_tb`: el *traceback* (objeto `traceback`)
    - Si no hubo excepción, los tres valen `None`.

In [ ]:
import time

class RegistroTiempo:
    """Context manager que mide tiempo de ejecución del bloque."""

    def __init__(self, nombre: str) -> None:
        self.nombre = nombre
        self.inicio: float | None = None

    def __enter__(self):
        self.inicio = time.perf_counter()
        print(f">>> Entrando en '{self.nombre}'")
        return self

    def __exit__(self, exc_type, exc_val, exc_tb):
        duracion = time.perf_counter() - self.inicio
        if exc_type is None:
            print(f"<<< '{self.nombre}' terminó en {duracion:.4f} s (sin errores)")
        else:
            print(f"<<< '{self.nombre}' falló con {exc_type.__name__}: {exc_val} ({duracion:.4f} s)")
        # Devuelve None -> la excepción se propaga

with RegistroTiempo("operación lenta") as rt:
    total = sum(range(1_000_000))
    print("  suma:", total)

print("---")

In [ ]:
# __exit__ captura la excepción pero (por defecto) la propaga
try:
    with RegistroTiempo("división"):
        x = 1 / 0
        print("  no llegamos aquí")
except ZeroDivisionError:
    print("ZeroDivisionError propagada fuera del with (se limpió el recurso)")

In [ ]:
# Si __exit__ devuelve True, suprime la excepción
class SilenciaErrores:
    def __enter__(self):
        return self
    def __exit__(self, exc_type, exc_val, exc_tb):
        if exc_type is not None:
            print(f"Suprimiendo {exc_type.__name__}: {exc_val}")
        return True   # suprime la excepción

with SilenciaErrores():
    raise ValueError("no me verás escapar")

print("El programa continuó sin error capturado.")

## 6. Callable y arithmetic

Python permite que tus objetos sean **llamables** (como funciones) y que soporten **operadores aritméticos**, además de combinar protocolos en una misma clase.

| Método | Función que activa |
|--------|--------------------|
| `__call__` | `obj(...)` lo trata como función |
| `__add__` | `obj + otro` |
| `__radd__` | `otro + obj` (cuando `otro` no lo soporta) |
| `__iadd__` | `obj += otro` (acumulador) |
| `__mul__` | `obj * otro` |

Aquí combinamos varios protocolos en una clase que se comporta como un acumulador numérico llamable, iterable y sumable.

In [ ]:
class Acumulador:
    """Acumula valores, es llamable, sumable, iterable y comparable."""
    def __init__(self) -> None:
        self._valores: list[int] = []

    def __call__(self, valor: int) -> "Acumulador":
        """Acumula y devuelve self para encadenar llamadas."""
        self._valores.append(valor)
        return self

    def __add__(self, otro: "Acumulador") -> "Acumulador":
        nuevo = Acumulador()
        nuevo._valores = self._valores + otro._valores
        return nuevo

    def __radd__(self, otro: int) -> "Acumulador":
        """Permite 0 + acumulador (útil en sum(iterable) con inicial 0)."""
        nuevo = Acumulador()
        nuevo._valores = [otro] + self._valores
        return nuevo

    def __len__(self) -> int:
        return len(self._valores)

    def __iter__(self):
        return iter(self._valores)

    def __repr__(self) -> str:
        return f"Acumulador({self._valores!r})"

acc = Acumulador()
acc(3)(7)                 # llamable y encadenable (__call__)
acc(5)
print("acc:", acc)
print("len:", len(acc))
print("lista:", list(acc))

otro = Acumulador()
otro(10)(20)
print("acc + otro:", acc + otro)      # __add__
print("0 + acc (radd):", 0 + acc)      # __radd__

In [ ]:
# Con __radd__ (para el valor inicial 0) podemos pasar los acumuladores a sum()
accA = Acumulador()
accA(1)(2)
accB = Acumulador()
accB(30)(40)

# Cada Acumulador es iterable; los combinamos con sum y 0 + ...
total = sum(Acumulador()(5)(6))   # itera sus valores
print("suma de un acumulador:", total)

print("valores combinados:", list(accA) + list(accB))

## Tabla de referencia: protocolos y sus métodos

| Protocolo | Métodos dunder | Sintaxis que habilita |
|-----------|----------------|-----------------------|
| Representación | `__repr__`, `__str__`, `__format__` | `repr()`, `str()`, `print`, f-strings, `format()` |
| Secuencia / contenedor | `__len__`, `__getitem__`, `__setitem__`, `__delitem__`, `__contains__` | `len()`, indexado, slicing, `in`, `del`, iteración legada |
| Iteración | `__iter__`, `__next__` (+ `StopIteration`) | `for`, `iter()`, `next()` |
| Igualdad y comparación | `__eq__`, `__lt__` (+ `total_ordering`) | `==`, `<`, `<=`, `>`, `>=` |
| Hash | `__hash__` | `hash()`, uso en `set`/`dict` |
| Truthiness | `__bool__`, `__len__` | `bool()`, `if obj:`, `obj or ...` |
| Context manager | `__enter__`, `__exit__` | `with` |
| Callable | `__call__` | `obj(...)` |
| Aritmética | `__add__`, `__radd__`, `__mul__`, `__iadd__`, ... | `+`, `*`, `+=`, ... |

> Recuerda: los protocolos son **contratos informales** (duck typing). No hay interfaz que implementar: basta con que los métodos existan y devuelvan lo esperado.

## Ejercicios

### Ejercicio 1 (Guiado): Representación

Crea una clase `Producto` con `nombre` y `precio`. Implementa `__repr__` (sin ambigüedad) y `__str__` (legible: `"nombre - $precio"`).

```python
class Producto:
    def __init__(self, nombre: str, precio: float) -> None:
        ...

    # implementa __repr__ y __str__

p = Producto("Café", 4.5)
print(str(p))   # "Café - $4.5"
print(repr(p))  # Producto('Café', 4.5)
```

**Solución** (copia y ejecuta):

In [ ]:
class Producto:
    def __init__(self, nombre: str, precio: float) -> None:
        self.nombre = nombre
        self.precio = precio

    def __repr__(self) -> str:
        return f"Producto({self.nombre!r}, {self.precio!r})"

    def __str__(self) -> str:
        return f"{self.nombre} - ${self.precio}"

p = Producto("Café", 4.5)
print(str(p))
print(repr(p))

### Ejercicio 2 (Guiado): Protocolo de secuencia

Implementa una clase `Banda` que guarde una lista de músicos y soporte `len()`, indexado, asignación, `in` y **iteración legada**.

**Solución** (copia y ejecuta):

In [ ]:
class Banda:
    def __init__(self, nombre: str, musicos: list[str]) -> None:
        self.nombre = nombre
        self._musicos = musicos

    def __len__(self) -> int:
        return len(self._musicos)

    def __getitem__(self, i: int | slice):
        return self._musicos[i]

    def __setitem__(self, i: int, valor: str) -> None:
        self._musicos[i] = valor

    def __contains__(self, musico: str) -> bool:
        return musico in self._musicos

    def __repr__(self) -> str:
        return f"Banda({self.nombre!r}, {self._musicos!r})"

b = Banda("Los Dunder", ["Ana", "Bea", "Carlos"])
print(len(b))                 # 3
print(b[0], b[1:3])           # Ana ['Bea', 'Carlos']
print("Ana" in b)             # True
b[0] = "Alicia"
for m in b:                   # iteración legada
    print(" *", m)

### Ejercicio 3 (Guiado): Iterador

Crea un iterador `Fibonacci` que genere los primeros `n` números de Fibonacci. Debe ser **iterable e iterador a la vez**. Comprueba que `list(Fibonacci(8))` funciona.

**Solución** (copia y ejecuta):

In [ ]:
class Fibonacci:
    def __init__(self, n: int) -> None:
        self.n = n
        self._contador = 0
        self._a, self._b = 0, 1

    def __iter__(self):
        return self

    def __next__(self) -> int:
        if self._contador >= self.n:
            raise StopIteration
        valor = self._a
        self._a, self._b = self._b, self._a + self._b
        self._contador += 1
        return valor

    def __repr__(self) -> str:
        return f"Fibonacci({self.n})"

print(list(Fibonacci(8)))  # [0, 1, 1, 2, 3, 5, 8, 13]
for x in Fibonacci(5):
    print(x, end=" ")
print()

### Ejercicio 4 (Independiente): Clase multi-protocolo

Diseña una clase `Caja` que implemente **varios protocolos a la vez**:

- Debe almacenar una lista de ítems.
- Protocolo de secuencia: `len()`, indexado y `in`.
- Protocolo de iteración: iterable (recorre sus ítems).
- Igualdad: dos cajas son iguales si tienen los mismos ítems (mismo orden).
- Hashable: `hash()` basado en los ítems (**pista**: conviértelos a tupla, si son hashables).
- Truthiness: una caja vacía es falsa.
- `__str__` legible y `__repr__` sin ambigüedad.

**Requisitos**: que `len(caja)`, `caja[0]`, `item in caja`, `for x in caja`, `caja1 == caja2`, `hash(caja)` y `bool(caja)` funcionen. Muestra un ejemplo de uso completo.

> Piensa: al definir `__iter__` (en lugar de depender solo de `__getitem__`), la iteración es explícita. ¿Qué implica definir `__eq__` respecto a `__hash__`?

**Solución sugerida** (compara la tuya):

In [ ]:
class Caja:
    def __init__(self, items: list) -> None:
        self._items = list(items)

    # Secuencia
    def __len__(self) -> int:
        return len(self._items)

    def __getitem__(self, i: int | slice):
        return self._items[i]

    def __contains__(self, item) -> bool:
        return item in self._items

    # Iteración explícita
    def __iter__(self):
        return iter(self._items)

    # Igualdad y hash
    def __eq__(self, otro) -> bool:
        if not isinstance(otro, Caja):
            return NotImplemented
        return self._items == otro._items

    def __hash__(self) -> int:
        # Necesitamos ítems hashables para poder hashear la tupla
        return hash(tuple(self._items))

    # Truthiness: vacía = falsa (se apoya en __len__)

    # Representación
    def __str__(self) -> str:
        return f"Caja({', '.join(map(str, self._items))})"

    def __repr__(self) -> str:
        return f"Caja({self._items!r})"

c1 = Caja([1, 2, 3])
c2 = Caja([1, 2, 3])
c3 = Caja([7])

print("len:", len(c1))
print("c1[0]:", c1[0])
print("2 in c1:", 2 in c1)
print("iteración:", [x * 10 for x in c1])
print("c1 == c2:", c1 == c2)
print("hash:", hash(c1))
print("bool vacía:", bool(Caja([])))
print("bool con items:", bool(c3))
print("str:", str(c1))
print("repr:", repr(c1))

In [ ]:
# Verificación final de Caja dentro de un conjunto (usa __hash__ y __eq__)
conjunto = {Caja([1, 2]), Caja([1, 2]), Caja([9])}
print("conjunto:", conjunto)

# La truthiness de Caja se apoya en __len__ (vacía -> falsa)
if Caja([]):
    print("debería ser falsa")
else:
    print("Caja vacía es falsa ✔")

## Resumen

1. **Métodos dunder** (`__método__`) son puntos de enganche que el intérprete invoca automáticamente: al escribir `len(x)` en realidad se llama a `x.__len__()`.
2. **Protocolos** son contratos informales basados en duck typing: basta con implementar los métodos esperados para que la clase se integre con la sintaxis del lenguaje.
3. **Representación**: `__repr__` (sin ambigüedad, para debug) vs `__str__` (legible, para humanos), y `__format__` para f-strings personalizadas.
4. **Secuencia**: `__len__` + `__getitem__` habilitan `len()`, indexado, `in` e incluso iteración legada automática.
5. **Iteración**: el protocolo `__iter__`/`__next__` distingue **iterables** (pueden recrearse recorridos) de **iteradores** (un solo uso); `StopIteration` finaliza el `for`.
6. **Igualdad/hash**: `__eq__` y `__hash__` van parejas; definir `__eq__` sin `__hash__` vuelve el objeto unhashable. `__bool__`/`__len__` controlan la *truthiness*.
7. **Context manager**: `__enter__`/`__exit__` dan soporte a `with` para gestión de recursos y control de excepciones.
8. **Callable y aritmética**: `__call__`, `__add__`, `__radd__`, etc. hacen los objetos usables como funciones y con operadores.

**Clave final**: los dunder no son magia, son **convenciones poderosas**. Cuando entiendes qué protocolo pide cada sintaxis de Python, puedes construir clases que se sienten como tipos nativos del lenguaje.